# Spark Structured Streaming

{doc}`Introduction to Stream Processing <01_introduction_to_stream_processing>`
described what stream processing is and why it is useful. This notebook
covers how Spark does it. **Structured Streaming** is Spark's stream
processing engine, and its central idea is that you should not have to learn
a second API: a streaming query is written with the same DataFrame
operations as a batch query, and Spark works out how to run it
incrementally.

## The Programming Model

Structured Streaming treats a stream as a table that keeps growing
([Apache Spark: Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/streaming/getting-started.html#programming-model)).
Every new record is a row appended to this **input table**. Your query is an
ordinary DataFrame query over the input table, and its output is a **result
table**. Each time the query is triggered, Spark processes the rows that
arrived since the last trigger and updates the result table, then writes
the changes to a **sink**.

```mermaid
:caption: Each trigger processes the newly arrived rows of the input table, updates the result table and writes the changes to the sink.
:align: center

graph LR
    Src(["Source: files, Kafka, a Delta table"]) --> In[("Input table, always growing")]
    In --> Q["Your DataFrame query, run incrementally"]
    Q --> Res[("Result table")]
    Res --> Sink(["Sink: Delta table, Kafka, ..."])
    Q -. "progress" .-> CP[("Checkpoint")]
```

The pieces you specify are:

1. A **source**, with `spark.readStream`.
2. **Transformations**, with the usual DataFrame API.
3. A **sink**, an **output mode** and a **trigger**, with `df.writeStream`.
4. A **checkpoint location**, where Spark stores its progress.

The rest of this notebook goes through them in that order.

## Setup: Simulating Arriving Data

The bakehouse sample tables are static, so we simulate a stream. A helper
function writes one day's transactions as JSON files into a "landing"
folder in a volume, the way an upstream system might drop a file per day.
Calling it again with the next day simulates new data arriving.

The setup cell also removes tables and checkpoints left over from an earlier
run, so the notebook can be run again from the top.

In [1]:
from pyspark.sql import functions as F

from project.utils.sparksession import get_spark_session

spark = get_spark_session(profile="free-edition")

spark.sql("DROP VOLUME IF EXISTS workspace.default.streaming_02")
spark.sql("CREATE VOLUME workspace.default.streaming_02")
for table in ["stream02_bronze", "stream02_silver", "stream02_product_counts", "stream02_scratch"]:
    spark.sql(f"DROP TABLE IF EXISTS workspace.default.{table}")

VOLUME = "/Volumes/workspace/default/streaming_02"
LANDING = f"{VOLUME}/landing"
CHECKPOINTS = f"{VOLUME}/checkpoints"

sales = spark.read.table("samples.bakehouse.sales_transactions")


def land_day(day: str) -> int:
    """Write one day of sales as JSON files into the landing folder."""
    daily = sales.filter(F.to_date("dateTime") == day).drop("cardNumber")
    daily.write.mode("overwrite").json(f"{LANDING}/{day}")
    return daily.count()

Land the first day:

In [2]:
land_day("2024-05-01")

207

## Sources

### Files

The simplest source is a folder of files. Spark watches the folder and
treats every new file as new rows. Unlike a batch read, a streaming file
source **requires a schema** by default, which ensures that the query uses a
consistent schema even across failures
([Apache Spark: Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/streaming/apis-on-dataframes-and-datasets.html)):

In [3]:
spark.readStream.json(f"{LANDING}/*").columns

IllegalArgumentException: [UNABLE_TO_INFER_SCHEMA] Unable to infer schema for JSON. It must be specified manually. SQLSTATE: 42KD9

JVM stacktrace:
org.apache.spark.SparkIllegalArgumentException
	at org.apache.spark.sql.errors.QueryExecutionErrors$.createStreamingSourceNotSpecifySchemaError(QueryExecutionErrors.scala:1182)
	at org.apache.spark.sql.execution.datasources.DataSource.sourceSchema(DataSource.scala:424)
	at org.apache.spark.sql.execution.datasources.DataSource.sourceInfo$lzycompute(DataSource.scala:188)
	at org.apache.spark.sql.execution.datasources.DataSource.sourceInfo(DataSource.scala:188)
	at org.apache.spark.sql.execution.streaming.runtime.StreamingRelation$.apply(StreamingRelation.scala:46)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource$$anonfun$apply$1.applyOrElse(ResolveDataSource.scala:305)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource$$anonfun$apply$1.applyOrElse(ResolveDataSource.scala:62)
	at org.apache.spark.sql.catalyst.plans.logica

The error asks for a schema. Declare one in DDL form:

In [4]:
SALES_SCHEMA = """
    transactionID BIGINT, customerID BIGINT, franchiseID BIGINT, dateTime TIMESTAMP,
    product STRING, quantity BIGINT, unitPrice BIGINT, totalPrice BIGINT, paymentMethod STRING
"""

raw_files = spark.readStream.schema(SALES_SCHEMA).json(f"{LANDING}/*")
raw_files.isStreaming

True

### Auto Loader

On Databricks, the recommended way to ingest files incrementally is **Auto
Loader**, the `cloudFiles` source, which processes new data files
incrementally and efficiently as they arrive
([What is Auto Loader?](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/)).
It can infer the schema from a sample of the files and evolve it when new
columns appear; setting `cloudFiles.schemaLocation` enables this and stores
the schema there, so it stays stable across runs. Data that doesn't match the
schema is kept as JSON in a `_rescued_data` column instead of being lost
([Configure schema inference and evolution in Auto Loader](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/schema)).

In [5]:
auto_loaded = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINTS}/bronze_schema")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaHints", "dateTime TIMESTAMP")
    .load(LANDING)
)

auto_loaded.printSchema()

root
 |-- customerID: long (nullable = true)
 |-- dateTime: timestamp (nullable = true)
 |-- franchiseID: long (nullable = true)
 |-- paymentMethod: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- totalPrice: long (nullable = true)
 |-- transactionID: long (nullable = true)
 |-- unitPrice: long (nullable = true)
 |-- _rescued_data: string (nullable = true)



For JSON, Auto Loader infers every column as a string unless
`cloudFiles.inferColumnTypes` is `true`, and even then JSON has no
timestamp type, so `dateTime` would stay a string. A **schema hint** enforces
the type you expect for selected columns while the rest is still inferred
([Configure schema inference and evolution in Auto Loader](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/schema#override-schema-inference-with-schema-hints)).

### Tables and Message Buses

A **Delta table** is also a streaming source: `spark.readStream.table(name)`
processes new records as new table versions are committed to the source
table. If existing records in the source are updated or deleted, the stream
fails, unless you set `skipChangeCommits`
([Delta table streaming reads and writes](https://docs.databricks.com/aws/en/structured-streaming/delta-lake#handle-changes-to-source-delta-lake-tables)). This is how
the layers of a medallion pipeline feed each other, as you will see below.

In production, events usually come from a message bus. Reading from Kafka
looks like this. It is not executed here, because it needs a Kafka cluster:

```python
events = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "broker-1:9092,broker-2:9092")
    .option("subscribe", "bakery-sales")
    .option("startingOffsets", "earliest")
    .load()
    .select(F.from_json(F.col("value").cast("string"), SALES_SCHEMA).alias("sale"))
    .select("sale.*")
)
```

Kafka delivers each message as binary `key` and `value` columns, which you
deserialize explicitly
([Stream processing with Apache Kafka and Databricks](https://docs.databricks.com/aws/en/connect/streaming/kafka/#schema-for-kafka-rows)),
so parsing the JSON payload with `from_json`, covered in
{doc}`Working with Complex Data <../02_developing_applications_with_apache_spark/04_working_with_complex_data>`,
is part of almost every Kafka pipeline.

## Transformations

A streaming DataFrame supports almost every transformation a static one
does. Operations that look at one row at a time are **stateless**, and work
exactly as in
{doc}`DataFrame API Basics <../02_developing_applications_with_apache_spark/01_dataframe_api_basics>`:

In [6]:
cleaned = (
    auto_loaded
    .filter((F.col("quantity") > 0) & (F.col("totalPrice") > 0))
    .withColumns(
        {
            "sale_date": F.to_date("dateTime"),
            "sale_hour": F.hour("dateTime"),
        }
    )
)

A stream can also be joined with a static DataFrame, to enrich each event
with reference data. This **stream-static join** is not stateful
([Apache Spark: Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/streaming/apis-on-dataframes-and-datasets.html)):
each micro-batch is joined with the static side as it is when the batch
runs. Because `franchises` is small,
broadcasting it avoids a shuffle, as explained in
{ref}`Join Strategies <join-strategies>`:

In [7]:
franchises = spark.read.table("samples.bakehouse.sales_franchises").select(
    "franchiseID",
    F.col("name").alias("franchise_name"),
    F.col("city").alias("franchise_city"),
    F.col("country").alias("franchise_country"),
)

enriched = cleaned.join(F.broadcast(franchises), "franchiseID", "left")

Operations that need to remember earlier rows, such as aggregations,
de-duplication and stream-stream joins, are **stateful**. They are the topic
of
{doc}`Stream Aggregations, Windows and Watermarks <03_stream_aggregations_windows_and_watermarks>`.
A few operations are not supported on streams at all, because they would
need the complete input: `limit`, `distinct`, sorting (except after an
aggregation in complete mode), and actions like `count` or `show`
([Apache Spark: Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/streaming/apis-on-dataframes-and-datasets.html)).

## Sinks, Output Modes and Triggers

`writeStream` turns a streaming DataFrame into a running query. It needs
three decisions, plus a checkpoint location.

### Sinks

The Delta transaction log guarantees exactly-once processing for a Delta
sink
([Delta table streaming reads and writes](https://docs.databricks.com/aws/en/structured-streaming/delta-lake#use-delta-lake-tables-as-a-sink)).
The guarantees of the other sinks come from the Spark sink table
([Apache Spark: Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/streaming/apis-on-dataframes-and-datasets.html)):

| Sink | Use | Notes |
| --- | --- | --- |
| Delta table, `toTable(name)` | Almost everything on Databricks | Exactly-once, transactional |
| Kafka, `format("kafka")` | Publishing events to other systems | At-least-once |
| `foreachBatch(func)` | Any batch writer, `MERGE`, writing to several tables | At-least-once; depends on your function |
| `foreach(writer)` | Custom per-row output | At-least-once |
| Memory, `format("memory")` | Debugging small streams | Not fault-tolerant |

`foreachBatch` deserves a special mention. It applies a batch function to
the output of every micro-batch
([Use foreachBatch to write to arbitrary data sinks](https://docs.databricks.com/aws/en/structured-streaming/foreach)),
passing an ordinary, static DataFrame plus the batch ID, so anything you can
do in batch, including `MERGE INTO` and writing to several tables, works
inside it. Here is an upsert into a table of latest sales per
customer. It is not executed here, because this course's free serverless
workspace cannot start the Python sandbox that `foreachBatch` needs:

```python
def upsert_latest_sale(batch_df, batch_id):
    batch_df.createOrReplaceTempView("updates")
    batch_df.sparkSession.sql("""
        MERGE INTO workspace.default.latest_sale_per_customer AS target
        USING (SELECT * FROM updates QUALIFY row_number() OVER (PARTITION BY customerID ORDER BY dateTime DESC) = 1) AS source
        ON target.customerID = source.customerID
        WHEN MATCHED AND source.dateTime > target.dateTime THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)


(
    enriched.writeStream
    .foreachBatch(upsert_latest_sale)
    .option("checkpointLocation", f"{CHECKPOINTS}/latest_sale")
    .trigger(availableNow=True)
    .start()
)
```

`foreachBatch` provides only at-least-once write guarantees, so a retried
batch can be written twice. Make the function idempotent, as the `MERGE`
above is, or, for Delta writes, use the `txnAppId` and `txnVersion` writer
options with `txnVersion` bound to the batch ID
([Use foreachBatch to write to arbitrary data sinks](https://docs.databricks.com/aws/en/structured-streaming/foreach)).

### Output Modes

The **output mode** decides which rows of the result table are written on
each trigger
([Apache Spark: Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/streaming/apis-on-dataframes-and-datasets.html)):

| Mode | Writes | Typical use |
| --- | --- | --- |
| `append` (default) | Only new rows, which will never change again | Stateless pipelines; windows closed by a watermark |
| `update` | Rows that changed since the last trigger | Aggregations written to a sink that can upsert |
| `complete` | The whole result table, every time | Small aggregations |

For a stateless query, every output row is final as soon as it is produced,
so `append` is the natural choice. For an aggregation, a group's value can
keep changing as data arrives, so `append` is not allowed unless a watermark
tells Spark when a group is final:

In [8]:
product_counts = auto_loaded.groupBy("product").agg(F.count("*").alias("transactions"))

(
    product_counts.writeStream
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINTS}/product_counts_append")
    .trigger(availableNow=True)
    .toTable("workspace.default.stream02_product_counts")
)

AnalysisException: [STREAMING_OUTPUT_MODE.UNSUPPORTED_OPERATION] Invalid streaming output mode: append. This output mode is not supported for streaming aggregations without watermark on streaming DataFrames/DataSets. SQLSTATE: 42KDE

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at org.apache.spark.sql.errors.QueryCompilationErrors$.unsupportedOutputModeForStreamingOperationError(QueryCompilationErrors.scala:3557)
	at org.apache.spark.sql.catalyst.analysis.UnsupportedOperationChecker$.checkForStreaming(UnsupportedOperationChecker.scala:303)
	at org.apache.spark.sql.execution.streaming.runtime.ResolveWriteToStream$$anonfun$apply$1.applyOrElse(ResolveWriteToStream.scala:77)
	at org.apache.spark.sql.execution.streaming.runtime.ResolveWriteToStream$$anonfun$apply$1.applyOrElse(ResolveWriteToStream.scala:48)
	at org.apache.spark.sql.catalyst.plans.logical.AnalysisHelper.$anonfun$resolveOperatorsDownWithPruning$2(AnalysisHelper.scala:208)
	at org.apache.spark.sql.catalyst.trees.Cur

The `AnalysisException` says that append output mode is not supported for
streaming aggregations without a watermark.

### Triggers

The trigger decides when micro-batches run
([Configure Structured Streaming trigger intervals](https://docs.databricks.com/aws/en/structured-streaming/triggers)):

| Trigger | Behaviour |
| --- | --- |
| Not specified | `processingTime` with an interval of 0: check for new data every few milliseconds |
| `processingTime="1 minute"` | Start a micro-batch at a fixed interval |
| `availableNow=True` | Process everything available now, in one or more micro-batches, then stop |
| `realTime="5 minutes"` | Real-time mode; see {doc}`Real-Time Mode in Structured Streaming <04_real_time_mode_in_structured_streaming>` |

The first two run forever, until the query is stopped. `availableNow` makes
a streaming query behave like an incremental batch job, and Databricks
recommends it for all incremental batch processing workloads
([Configure Structured Streaming trigger intervals](https://docs.databricks.com/aws/en/structured-streaming/triggers#availablenow-incremental-batch-processing)):
schedule it as a job every hour, and each run processes only what arrived in
the last hour. On serverless compute only `availableNow` and the deprecated
`once` trigger are supported
([Configure Structured Streaming trigger intervals](https://docs.databricks.com/aws/en/structured-streaming/triggers#trigger-modes-overview)),
which is why all queries in this chapter use `availableNow`:

In [9]:
(
    auto_loaded.writeStream
    .option("checkpointLocation", f"{CHECKPOINTS}/never_used")
    .trigger(processingTime="1 minute")
    .toTable("workspace.default.stream02_scratch")
)

AnalysisException: [INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED] Trigger type ProcessingTime is not supported for this cluster type.
Use a different trigger type e.g. AvailableNow, Once. SQLSTATE: 0A000

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at org.apache.spark.sql.errors.QueryCompilationErrors$.infiniteStreamingTriggerNotSupportedError(QueryCompilationErrors.scala:7660)
	at org.apache.spark.sql.streaming.DataStreamUtils$.checkTriggerSupportedOnServerless(DataStreamUtils.scala:601)
	at org.apache.spark.sql.classic.streaming.startInternal(DataStreamWriter.scala:523)
	at org.apache.spark.sql.classic.streaming.writeToV1Table$1(DataStreamWriter.scala:330)
	at org.apache.spark.sql.classic.streaming.$anonfun$toTable$1(DataStreamWriter.scala:365)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.sql.classic.streaming.toTable(DataStreamWriter.scala:231)
	at org.apache.spark.sql.classic.streaming.toTable(DataStreamWriter.scala:90)
	at org.apache.spark.sql.connect.

The error is `INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED`.

## A Streaming Pipeline with Checkpoints

Time to put it together: a two-step medallion pipeline, as in
{doc}`ETL with the DataFrame API <../01_introduction_to_apache_spark/04_etl_with_the_dataframe_api/index>`,
but incremental. Auto Loader ingests the landed files into a bronze table,
and a second stream reads the bronze table, cleans and enriches the rows and
appends them to a silver table.

In [10]:
def run_bronze():
    query = (
        auto_loaded
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .writeStream
        .option("checkpointLocation", f"{CHECKPOINTS}/bronze")
        .trigger(availableNow=True)
        .toTable("workspace.default.stream02_bronze")
    )
    query.awaitTermination()
    return query


def run_silver():
    bronze = spark.readStream.table("workspace.default.stream02_bronze")
    silver = (
        bronze
        .filter((F.col("quantity") > 0) & (F.col("totalPrice") > 0))
        .withColumns({"sale_date": F.to_date("dateTime"), "sale_hour": F.hour("dateTime")})
        .join(F.broadcast(franchises), "franchiseID", "left")
        .drop("_rescued_data")
    )
    query = (
        silver.writeStream
        .option("checkpointLocation", f"{CHECKPOINTS}/silver")
        .trigger(availableNow=True)
        .toTable("workspace.default.stream02_silver")
    )
    query.awaitTermination()
    return query


def rows_processed(query) -> int:
    return sum(p["sources"][0]["numInputRows"] for p in query.recentProgress)

Each query must have its **own** checkpoint location; multiple queries
should never share one
([Structured Streaming checkpoints](https://docs.databricks.com/aws/en/structured-streaming/checkpoints)).

Run both steps. Only the first day has landed so far:

In [11]:
bronze_query = run_bronze()
silver_query = run_silver()

rows_processed(bronze_query), rows_processed(silver_query)

(207, 207)

Now a second and a third day of sales arrive, and the pipeline runs again:

In [12]:
land_day("2024-05-02"), land_day("2024-05-03")

(190, 204)

In [13]:
bronze_query = run_bronze()
silver_query = run_silver()

rows_processed(bronze_query), rows_processed(silver_query)

(394, 394)

The second run processed only the two new days. A third run, with no new
files, processes nothing at all:

In [14]:
rows_processed(run_bronze()), rows_processed(run_silver())

(0, 0)

The silver table now holds the three days exactly once, enriched with the
franchise details:

In [15]:
silver_table = spark.read.table("workspace.default.stream02_silver")

silver_table.groupBy("sale_date").count().orderBy("sale_date").show()
silver_table.select("transactionID", "dateTime", "product", "totalPrice", "franchise_name", "franchise_city").show(3)

+----------+-----+
| sale_date|count|
+----------+-----+
|2024-05-01|  207|
|2024-05-02|  190|
|2024-05-03|  204|
+----------+-----+



+-------------+--------------------+------------------+----------+----------------+--------------+
|transactionID|            dateTime|           product|totalPrice|  franchise_name|franchise_city|
+-------------+--------------------+------------------+----------+----------------+--------------+
|      1000850|2024-05-03 19:01:...|Golden Gate Ginger|        72|Sweet Sinsations|     Stockholm|
|      1002760|2024-05-03 08:43:...|     Orchard Oasis|        36|Sweet Sinsations|     Stockholm|
|      1000477|2024-05-03 07:25:...|     Tokyo Tidbits|        24|Sweet Sinsations|     Stockholm|
+-------------+--------------------+------------------+----------+----------------+--------------+
only showing top 3 rows


:::::{dropdown} Example: Reconciling the Pipeline with the Source
:open:
:color: primary
:icon: pencil
:class-body: example-body

The first run ingested 207 rows. The second run ingested the two new days,
190 and 204 rows. Does the silver table contain exactly the sales of the
first three days of May?

**Add up what each run processed.**

$$
207 + (190 + 204) = 601
$$

**Count the same days in the source table.**

```python
sales.filter(F.to_date("dateTime") <= "2024-05-03").count()
```

Both sides equal 601, so no sale was lost and none was processed twice,
even though the queries ran three times.
:::::

Checking it:

In [16]:
sales.filter(F.to_date("dateTime") <= "2024-05-03").count() == silver_table.count()

True

### What the Checkpoint Contains

The checkpoint is an ordinary folder that tracks the information
identifying the query, including state and processed records, in `offsets`,
`commits`, `state` and `metadata`
([Structured Streaming checkpoints](https://docs.databricks.com/aws/en/structured-streaming/checkpoints)).
Listing the silver query's checkpoint shows them:

In [17]:
spark.sql(f"LIST '{CHECKPOINTS}/silver'").select("name").show()

+--------+
|    name|
+--------+
|commits/|
|metadata|
|offsets/|
+--------+



- `metadata` holds the query's unique ID, which survives restarts.
- `offsets/` holds one file per micro-batch, written **before** the batch
  runs, recording which part of the source the batch covers. For a Delta
  source this is a table version.
- `commits/` holds one file per micro-batch, written **after** the batch
  has been committed to the sink.
- `state/`, for stateful queries only, holds the state store.

Spark records the offset range of each trigger before processing it
([Apache Spark: Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/streaming/getting-started.html#fault-tolerance-semantics)),
so on restart a batch that was planned but never committed is re-run with
exactly the same offsets. Together with an idempotent sink such as Delta,
this gives exactly-once results.

:::{admonition} Note: Checkpoints Belong to a Query
:class: note

A checkpoint is tied to the query that created it. You can change some
things between runs, such as adding or removing filters, rate limits or the
trigger interval, but changes to the number or type of input sources, the
stateful operations or the output sink type generally need a new checkpoint.
With a new or deleted checkpoint, the next run begins fresh
([Structured Streaming checkpoints](https://docs.databricks.com/aws/en/structured-streaming/checkpoints#recover-after-changes-in-a-structured-streaming-query)).
To reprocess everything from the start, delete the checkpoint and the target
table together, as the setup cell of this notebook does.
:::

## Monitoring a Query

`writeStream...start()` and `toTable()` return a `StreamingQuery` object. It
identifies the query and reports what it is doing:

In [18]:
silver_query.id, silver_query.status

('8a10b998-bb59-4a0f-9615-6017b206ae3c',
 {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False})

`recentProgress` holds a progress report for each recent micro-batch, and
`lastProgress` the latest one. The report says how many rows the batch
read (`numInputRows`), how long each phase took (`durationMs`) and which
source offsets it covered
([Monitoring Structured Streaming queries](https://docs.databricks.com/aws/en/structured-streaming/stream-monitoring#streamingquerylistener-object-metrics)). Here is
the first micro-batch of the bronze query's second run, the one that
ingested two days of files:

In [19]:
import json

progress = bronze_query.recentProgress[0]
summary = {key: progress[key] for key in ["batchId", "timestamp", "batchDuration", "durationMs"]}
summary["numInputRows"] = progress["sources"][0]["numInputRows"]
print(json.dumps(summary, indent=2))

{
  "batchId": 1,
  "timestamp": "2026-10-12T20:55:08.330Z",
  "batchDuration": 4828,
  "durationMs": {
    "triggerExecution": 4826,
    "queryPlanning": 277,
    "collectSourceMetrics": 1,
    "getBatch": 11,
    "commitOffsets": 75,
    "postCommitBatchCallback": 421,
    "addBatch": 1152,
    "latestOffset": 1414,
    "postAddBatchCallback": 375,
    "commitBatch": 230,
    "walCommit": 217
  },
  "numInputRows": 394
}


For long-running queries, `spark.streams.active` lists all running queries
of the session, `query.stop()` stops one, and `query.awaitTermination()`
blocks until it ends, re-raising any error that stopped it. To push the same
metrics to an external monitoring service, register a
`StreamingQueryListener`, and on classic compute the Spark UI shows them on
its **Streaming** tab
([Monitoring Structured Streaming queries](https://docs.databricks.com/aws/en/structured-streaming/stream-monitoring)).

## Cleaning Up

In [20]:
for table in ["stream02_bronze", "stream02_silver", "stream02_product_counts", "stream02_scratch"]:
    spark.sql(f"DROP TABLE IF EXISTS workspace.default.{table}")
spark.sql("DROP VOLUME IF EXISTS workspace.default.streaming_02");

## Choosing Sources, Sinks and Triggers

| Decision | Default choice on Databricks | Alternative |
| --- | --- | --- |
| Ingesting files | Auto Loader (`cloudFiles`) | The plain file source, with a declared schema, for simple cases |
| Passing data between pipeline layers | A Delta table as both sink and source | `foreachBatch` when a layer needs `MERGE` or several outputs |
| Trigger | `availableNow`, scheduled as a job | `processingTime` on classic compute for second-level latency; real-time mode for milliseconds |
| Output mode | `append` for stateless queries | `update` or `complete` for aggregations, or `append` with a watermark |

## Common Mistakes

- **Sharing a checkpoint location between queries.** Each query needs its
  own
  ([Structured Streaming checkpoints](https://docs.databricks.com/aws/en/structured-streaming/checkpoints)).
- **Reading a file stream without a schema.** The plain file source
  requires one by default, as the error above showed. Use Auto Loader with a
  `cloudFiles.schemaLocation` when the schema should be inferred.
- **Updating or deleting rows in a Delta table that a stream reads.** The
  stream fails on the next run unless it sets `skipChangeCommits`
  ([Delta table streaming reads and writes](https://docs.databricks.com/aws/en/structured-streaming/delta-lake#handle-changes-to-source-delta-lake-tables)).
- **Writing an aggregation in append mode without a watermark.** Use
  `update` or `complete`, or add a watermark, as shown in the next notebook.
- **Assuming `foreachBatch` is exactly-once.** It is at-least-once; make
  the function idempotent
  ([Use foreachBatch to write to arbitrary data sinks](https://docs.databricks.com/aws/en/structured-streaming/foreach)).
- **Changing the query's sources or stateful operations and keeping the old
  checkpoint.** The restarted query is likely to fail with unpredictable
  errors
  ([Structured Streaming checkpoints](https://docs.databricks.com/aws/en/structured-streaming/checkpoints#recover-after-changes-in-a-structured-streaming-query)).

Next, {doc}`Stream Aggregations, Windows and Watermarks <03_stream_aggregations_windows_and_watermarks>`
adds state to streaming queries.

## Resources

- [Structured Streaming Programming Guide (Apache Spark)](https://spark.apache.org/docs/latest/streaming/index.html)
- [What is Auto Loader? (Databricks)](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/)
- [Configure Structured Streaming trigger intervals (Databricks)](https://docs.databricks.com/aws/en/structured-streaming/triggers)
- [Structured Streaming checkpoints (Databricks)](https://docs.databricks.com/aws/en/structured-streaming/checkpoints)
- [Use foreachBatch to write to arbitrary data sinks (Databricks)](https://docs.databricks.com/aws/en/structured-streaming/foreach)
- [Monitoring Structured Streaming queries (Databricks)](https://docs.databricks.com/aws/en/structured-streaming/stream-monitoring)